# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RAJkumar5736602/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #1 (growing content is longer & younger):** label = 30d vs prev-30d impression change; validation is a cohort comparison of means — directional, but confounded by client (big clients write long pages). I re-check it *within client* below. **Competition finding (LOW 62% more likely to grow):** same label; no grouping control. Constructive question: do these hold after controlling for client?

In [1]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p/'data/raw').exists())
sys.path.insert(0, str(root/'work/scripts'))
from wk import *
import pandas as pd, numpy as np
df = load()
g = df[df.trend_direction.isin(['up','down'])]
print('median words up vs down (pooled):', g.groupby('trend_direction').word_count.median().to_dict())
w = g.groupby(['client_id','trend_direction']).word_count.median().unstack().dropna()
print('clients where up-pages are longer (median):', int((w['up']>w['down']).sum()), 'of', len(w))

median words up vs down (pooled): {'down': 2909.0, 'up': 2847.5}
clients where up-pages are longer (median): 18 of 30


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Random row split vs client-holdout split, same RF.

In [2]:
X = features(df); y = df.y.values
out = {}
for name, (tr,te) in {'random_rows': random_split(df), 'client_holdout': client_split(df)}.items():
    m = models()['random_forest'].fit(X.iloc[tr], y[tr]); out[name] = evaluate(y[te], m.predict_proba(X.iloc[te])[:,1])
print(pd.DataFrame(out).T)
import json; json.dump(out, open(OUT/'split_comparison.json','w'), indent=1)

                roc_auc  avg_precision  p@50  p@200  base_rate       n
random_rows       0.769          0.782  0.94  0.925      0.542  6000.0
client_holdout    0.626          0.617  0.74  0.680      0.511  6163.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Final feature set contains no label or trend-window fields.

In [3]:
bad = [c for c in X.columns if any(k in c for k in ['trend','last_30','prev_30','client_id','content_id'])]
print('suspicious columns:', bad); assert not bad

suspicious columns: []


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Bold:** 'The model finds the pages that will decline.' **Rewrite:** 'On held-out clients, the model's top-50 review list contained a higher measured share of currently-declining pages than my hand rule; this is decision-support for ordering reviews, not a forecast or a causal claim.'

In [4]:
print('claim rewritten')

claim rewritten


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.